# QuantIQ | Phase 2 — Market Analysis

**Phase:** 2 | **Weeks:** 4–6 | **Deadline:** 14 June 2026

**Purpose:** Individual stock analysis for 12 NIFTY50 stocks — technical indicators, fundamentals, and cross-sector correlation.

---

## Team

| Handle | Stock          | Sector                |
|--------|----------------|-----------------------|
| AJ     | RELIANCE.NS    | Energy / Conglomerate |
| AV     | TCS.NS         | IT                    |
| AK     | INFY.NS        | IT                    |
| SS     | HCLTECH.NS     | IT                    |
| AR     | HDFCBANK.NS    | Banking               |
| EB     | ICICIBANK.NS   | Banking               |
| ShS    | AXISBANK.NS    | Banking               |
| RS     | TVSMOTOR.NS         | Auto                  |
| GT     | M&M.NS         | Auto                  |
| RT     | LT.NS          | Infrastructure        |
| NS     | TITAN.NS       | Consumer              |
| SmS    | APOLLOMICRO.NS | Defence               |

**Section 13 — Correlation Heatmap:** RS + GT

---

*Created by RS (Project Lead). Do not modify header cells or shared helper cells.*

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import yfinance as yf
from ta.trend import EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange
from src.data import fetch_ohlc

pio.templates["plotly_dark"].layout.paper_bgcolor = "#252526"
pio.templates["plotly_dark"].layout.plot_bgcolor  = "#1F2531"
pio.templates.default = "plotly_dark"

print("Imports OK")

Imports OK


In [2]:
TICKERS = {
    "AJ":  "RELIANCE.NS",
    "AV":  "TCS.NS",
    "AK":  "INFY.NS",
    "SS":  "HCLTECH.NS",
    "AR":  "HDFCBANK.NS",
    "EB":  "ICICIBANK.NS",
    "ShS": "AXISBANK.NS",
    "RS":  "TVSMOTOR.NS",
    "GT":  "M&M.NS",
    "RT":  "LT.NS",
    "NS":  "TITAN.NS",
    "SmS": "APOLLOMICRO.NS",
}

PERIOD   = "1y"
INTERVAL = "1d"

print(f"Config loaded — {len(TICKERS)} tickers, period={PERIOD}, interval={INTERVAL}")

Config loaded — 12 tickers, period=1y, interval=1d


In [3]:
def fetch_stock(
    ticker: str,
    period: str = PERIOD,
    interval: str = INTERVAL,
) -> pd.DataFrame:
    """Fetch OHLCV for a .NS ticker via fetch_ohlc.

    Args:
        ticker (str): NSE ticker with .NS suffix (e.g. "RELIANCE.NS").
        period (str): yfinance period string. Default "1y".
        interval (str): yfinance interval string. Default "1d".

    Returns:
        pd.DataFrame: OHLCV DataFrame indexed by datetime, NaN rows dropped.

    Raises:
        AssertionError: If ticker does not end with .NS.
        ValueError: If no data returned for ticker.
    """
    assert ticker.endswith(".NS"), f"Ticker must have .NS suffix, got: {ticker}"
    df = fetch_ohlc(ticker, period=period, interval=interval, use_cache=True)
    if df is None or df.empty:
        raise ValueError(f"No data returned for {ticker}")
    return df.dropna()

## Energy

## Section 1 — RELIANCE.NS | AJ (Energy / Conglomerate)

In [ ]:
# ── 1a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AJ"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 1b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 1c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 1d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 1e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 1f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 1g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AJ"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (AJ): Write 3–5 paragraphs on what the technical and fundamental data reveals about RELIANCE.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## IT

## Section 2 — TCS.NS | AV (IT)

In [ ]:
# ── 2a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AV"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 2b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 2c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 2d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 2e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 2f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 2g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AV"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (AV): Write 3–5 paragraphs on what the technical and fundamental data reveals about TCS.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 3 — INFY.NS | AK (IT)

In [ ]:
# ── 3a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AK"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 3b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 3c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 3d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 3e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 3f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 3g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AK"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (AK): Write 3–5 paragraphs on what the technical and fundamental data reveals about INFY.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 4 — HCLTECH.NS | SS (IT)

In [4]:
# ── 4a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["SS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HCLTECH.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,1190.099976,1212.800049,1190.099976,1195.099976,5970435
2026-06-02,1210.000000,1257.000000,1202.199951,1243.500000,7663067
2026-06-03,1228.800049,1230.000000,1176.800049,1179.000000,4800922
2026-06-04,1166.400024,1176.500000,1158.199951,1168.300049,2025595
2026-06-05,1179.000000,1182.500000,1147.000000,1154.699951,1971395


In [5]:
# ── 4b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,250.00
mean,1458.24,1472.23,1442.47,1456.64,3185349.53
std,149.19,148.60,148.11,149.59,2728440.64
min,1121.90,1143.20,1103.40,1124.00,0.00
25%,1371.00,1381.66,1347.46,1364.79,1931098.25
50%,1448.70,1459.13,1436.08,1445.27,2566080.00
75%,1589.53,1610.73,1580.25,1594.84,3456803.25
max,1746.56,1746.66,1668.56,1697.11,33066256.00


In [6]:
# ── 4c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [7]:
# ── 4d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [8]:
# ── 4e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [9]:
# ── 4f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [10]:
# ── 4g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["SS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 18.83
  EPS Growth (TTM) : -0.20%
  Debt / Equity    : 6.93
  Total Cash       : ₹3,204,999,936
  Total Debt       : ₹550,000,000
  Current Ratio    : 2.22
  PEG Ratio        : 2.39
  Dividend Yield   : 831.00%


### Observations

_TODO (SS): Write 3–5 paragraphs on what the technical and fundamental data reveals about HCLTECH.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Banking

## Section 5 — HDFCBANK.NS | AR (Banking)

In [ ]:
# ── 5a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AR"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 5b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 5c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 5d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 5e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 5f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 5g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AR"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (AR): Write 3–5 paragraphs on what the technical and fundamental data reveals about HDFCBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 6 — ICICIBANK.NS | EB (Banking)

In [ ]:
# ── 6a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["EB"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 6b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 6c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 6d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 6e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 6f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 6g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["EB"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (EB): Write 3–5 paragraphs on what the technical and fundamental data reveals about ICICIBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 7 — AXISBANK.NS | ShS (Banking)

In [ ]:
# ── 7a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["ShS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 7b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 7c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 7d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 7e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 7f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 7g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["ShS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (ShS): Write 3–5 paragraphs on what the technical and fundamental data reveals about AXISBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Auto

## Section 8 — TVSMOTOR.NS | RS (Auto)

In [ ]:
# ── 8a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["RS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 8b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 8c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 8d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 8e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 8f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 8g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["RS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (RS): Write 3–5 paragraphs on what the technical and fundamental data reveals about TVSMOTOR.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 9 — M&M.NS | GT (Auto)

In [ ]:
# ── 9a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["GT"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 9b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 9c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 9d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 9e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 9f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 9g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["GT"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (GT): Write 3–5 paragraphs on what the technical and fundamental data reveals about M&M.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Infrastructure

## Section 10 — LT.NS | RT (Infrastructure)

In [ ]:
# ── 10a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["RT"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 10b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 10c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 10d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 10e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 10f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 10g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["RT"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (RT): Write 3–5 paragraphs on what the technical and fundamental data reveals about LT.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Consumer

## Section 11 — TITAN.NS | NS (Consumer)

In [ ]:
# ── 11a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["NS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 11b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 11c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 11d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 11e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 11f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 11g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["NS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (NS): Write 3–5 paragraphs on what the technical and fundamental data reveals about TITAN.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Defence

## Section 12 — APOLLOMICRO.NS | SmS (Defence)

In [ ]:
# ── 12a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["SmS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 12b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 12c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 12d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 12e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 12f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 12g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["SmS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (SmS): Write 3–5 paragraphs on what the technical and fundamental data reveals about APOLLOMICRO.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 13 — Cross-Sector Correlation Heatmap | RS + GT

In [ ]:
# ── Section 13: Daily Return Correlation — RS + GT ──────────────────────────
# Uncomment and run after all 12 section data cells have been executed.

# from src.data import fetch_batch
#
# batch    = fetch_batch(list(TICKERS.values()), period=PERIOD, interval=INTERVAL)
# closes   = {
#     handle: batch[ticker]["Close"]
#     for handle, ticker in TICKERS.items()
#     if batch.get(ticker) is not None
# }
# close_df = pd.DataFrame(closes).dropna()
# returns  = close_df.pct_change().dropna()
# corr     = returns.corr()
#
# fig = px.imshow(
#     corr,
#     text_auto=".2f",
#     color_continuous_scale="RdBu_r",
#     zmin=-1, zmax=1,
#     title="NIFTY50 Sample — Daily Return Correlation (1Y)",
# )
# fig.update_layout(height=600)
# fig.show()

### Observations

_TODO (RS + GT): Write 3–5 paragraphs on correlation patterns — which sectors move together, which are uncorrelated, and what this implies for portfolio diversification._